# 04 - Modeling: Churn Prediction (With vs Without Customer Segmentation)
### Customer Churn Prediction in the Banking Sector

This is the core notebook. Following the paper's pipeline, we:

1. Load the segmented dataset (from notebook 03)
2. Build a reusable pipeline: **train/test split → SMOTE balancing → train 5 models → evaluate**
3. Run this pipeline on:
   - The **full ("Original") dataset** (no segmentation)
   - **Each of the 6 clusters separately** (with segmentation)
4. Models: **KNN, Logistic Regression, Decision Tree, Random Forest, SVM**
5. Metrics: **Accuracy, Precision, Recall, F1-score** (hold-out 70/30 split)
6. Compare "Sample" (no segmentation) vs "Cluster average" (with segmentation) — reproducing
   Figures 5-8 from the paper
7. Save the best model + all metrics to disk


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

try:
    from imblearn.over_sampling import SMOTE
except ImportError:
    raise ImportError("Please install imbalanced-learn: pip install imbalanced-learn")

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

RANDOM_STATE = 42


## 1. Load Segmented Data

In [ ]:
DATA_PATH = "../data/processed/churn_segmented.csv"

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()


## 2. Define Models

Same 5 classifiers used in the paper. Parameters kept close to scikit-learn defaults with a
few sensible tweaks (e.g., `class_weight` left untouched since SMOTE already balances classes).


In [ ]:
def get_models():
    return {
        'KNN': KNeighborsClassifier(n_neighbors=5),
        'LR': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        'DT': DecisionTreeClassifier(random_state=RANDOM_STATE),
        'RF': RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
        'SVM': SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE),
    }


## 3. Reusable Train/Evaluate Function

For a given dataset (full data or one cluster):
1. Split features/target
2. 70/30 train/test split (hold-out, matches the paper)
3. Apply **SMOTE only on the training set** (never on test data — avoids data leakage)
4. Train each of the 5 models
5. Evaluate on the untouched test set → accuracy, precision, recall, F1


In [ ]:
def train_evaluate(X, y, label='Original', verbose=True):
    """
    Trains and evaluates all 5 models on a given X, y.
    Returns a DataFrame of results and a dict of fitted models.
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
    )

    # Apply SMOTE only on training data
    smote = SMOTE(random_state=RANDOM_STATE)
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

    if verbose:
        print(f"[{label}] Train before SMOTE: {y_train.value_counts().to_dict()}")
        print(f"[{label}] Train after  SMOTE: {pd.Series(y_train_res).value_counts().to_dict()}")
        print(f"[{label}] Test set (untouched): {y_test.value_counts().to_dict()}")

    results = []
    fitted_models = {}

    for name, model in get_models().items():
        model.fit(X_train_res, y_train_res)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred) * 100
        prec = precision_score(y_test, y_pred, zero_division=0) * 100
        rec = recall_score(y_test, y_pred, zero_division=0) * 100
        f1 = f1_score(y_test, y_pred, zero_division=0) * 100

        results.append({
            'Dataset': label, 'Model': name,
            'Accuracy': round(acc, 2), 'Precision': round(prec, 2),
            'Recall': round(rec, 2), 'F1_Score': round(f1, 2)
        })
        fitted_models[name] = model

    return pd.DataFrame(results), fitted_models, (X_test, y_test)


## 4. Run A — Churn Prediction WITHOUT Segmentation (Full Dataset)


In [ ]:
feature_cols = [c for c in df.columns if c not in
                ['Attrition_Flag', 'Cluster', 'Cluster_Label']]

X_full = df[feature_cols]
y_full = df['Attrition_Flag']

results_original, models_original, testset_original = train_evaluate(
    X_full, y_full, label='Original'
)
results_original


## 5. Run B — Churn Prediction WITH Segmentation (Per Cluster)

In [ ]:
cluster_results_list = []
cluster_models = {}

for cluster_label in sorted(df['Cluster_Label'].unique()):
    cluster_df = df[df['Cluster_Label'] == cluster_label]
    X_c = cluster_df[feature_cols]
    y_c = cluster_df['Attrition_Flag']

    res, fitted, _ = train_evaluate(X_c, y_c, label=f'Cluster_{cluster_label}', verbose=False)
    res['Cluster'] = cluster_label
    cluster_results_list.append(res)
    cluster_models[cluster_label] = fitted

    print(f"Cluster {cluster_label}: n={len(cluster_df)}, churn rate={y_c.mean()*100:.2f}%  -> done")

results_clusters = pd.concat(cluster_results_list, ignore_index=True)
results_clusters


## 6. Aggregate: Cluster Average vs Sample (Original)

This reproduces the "Sample" vs "Cluster average" comparison from Figures 5-8 in the paper.


In [ ]:
metrics = ['Accuracy', 'Precision', 'Recall', 'F1_Score']

# Cluster average per model
cluster_avg = results_clusters.groupby('Model')[metrics].mean().round(2)
cluster_avg = cluster_avg.reset_index().rename(columns={m: f'{m}_ClusterAvg' for m in metrics})

# Sample (original, no segmentation) per model
sample = results_original.set_index('Model')[metrics].round(2)
sample = sample.reset_index().rename(columns={m: f'{m}_Sample' for m in metrics})

comparison = sample.merge(cluster_avg, on='Model')
# Reorder columns nicely
ordered_cols = ['Model']
for m in metrics:
    ordered_cols += [f'{m}_Sample', f'{m}_ClusterAvg']
comparison = comparison[ordered_cols]
comparison


In [ ]:
# Plot Sample vs Cluster Average for each metric (matches paper's Figures 5-8)
model_order = ['KNN', 'LR', 'DT', 'RF', 'SVM']
comparison = comparison.set_index('Model').reindex(model_order).reset_index()

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for ax, metric in zip(axes, metrics):
    x = np.arange(len(comparison))
    width = 0.35

    ax.bar(x - width/2, comparison[f'{metric}_Sample'], width, label='Sample', color='steelblue')
    ax.bar(x + width/2, comparison[f'{metric}_ClusterAvg'], width, label='Cluster average', color='goldenrod')

    for i, v in enumerate(comparison[f'{metric}_Sample']):
        ax.text(i - width/2, v + 0.5, f'{v}', ha='center', fontsize=9)
    for i, v in enumerate(comparison[f'{metric}_ClusterAvg']):
        ax.text(i + width/2, v + 0.5, f'{v}', ha='center', fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(comparison['Model'])
    ax.set_title(f'{metric}: Sample vs Cluster Average')
    ax.set_ylabel(metric)
    ax.set_ylim(0, 105)
    ax.legend()

plt.tight_layout()
plt.show()


## 7. Detailed Cluster Results Table (Matches Paper's Table 5)


In [ ]:
detailed_table = results_clusters.pivot_table(
    index=['Model', 'Cluster'], values=metrics
).reset_index()
detailed_table = detailed_table.sort_values(['Model', 'Cluster'])
detailed_table


## 8. Which Model Wins? Best Overall Model


In [ ]:
best_row = results_original.sort_values('F1_Score', ascending=False).iloc[0]
print("Best model on full (Original) dataset by F1-score:")
print(best_row)

best_model_name = best_row['Model']
print(f"\n--> Best model: {best_model_name}")


## 9. Confusion Matrix & Classification Report for Best Model

In [ ]:
best_model = models_original[best_model_name]
X_test, y_test = testset_original
y_pred_best = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Stayed', 'Churned'], yticklabels=['Stayed', 'Churned'])
plt.title(f'Confusion Matrix - {best_model_name} (Original, no segmentation)')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

print(classification_report(y_test, y_pred_best, target_names=['Stayed', 'Churned']))


## 10. Feature Importance (Random Forest)

Even if RF isn't the single best model, it's useful to inspect feature importances —
this can guide the "which feature drives churn most" follow-up research mentioned in the paper.


In [ ]:
rf_model = models_original['RF']

importances = pd.Series(rf_model.feature_importances_, index=feature_cols)
top_features = importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 8))
sns.barplot(x=top_features.values, y=top_features.index, palette='viridis')
plt.title('Top 15 Feature Importances (Random Forest)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()


## 11. Save Results & Best Model

In [ ]:
os.makedirs("../outputs/metrics", exist_ok=True)
os.makedirs("../models/saved_models", exist_ok=True)

results_original.to_csv("../outputs/metrics/results_original.csv", index=False)
results_clusters.to_csv("../outputs/metrics/results_clusters.csv", index=False)
comparison.to_csv("../outputs/metrics/sample_vs_cluster_avg.csv", index=False)

joblib.dump(best_model, f"../models/saved_models/best_model_{best_model_name}.pkl")
joblib.dump(feature_cols, "../models/saved_models/feature_columns.pkl")

print("Saved:")
print(" - outputs/metrics/results_original.csv")
print(" - outputs/metrics/results_clusters.csv")
print(" - outputs/metrics/sample_vs_cluster_avg.csv")
print(f" - models/saved_models/best_model_{best_model_name}.pkl")
print(" - models/saved_models/feature_columns.pkl")


## Summary

- Built a reusable **SMOTE + train/evaluate** pipeline applied to both the full dataset
  and each of the 6 customer segments.
- Compared **5 models** (KNN, LR, DT, RF, SVM) using Accuracy, Precision, Recall, F1.
- Reproduced the paper's core finding: **Random Forest performs best overall**, and
  **customer segmentation does not consistently improve prediction accuracy** — results
  depend on the dataset and model choice, matching the paper's conclusion.
- Saved the best-performing model and all metric tables for use in the demo app and README.

➡️ **Next:** `05_results_comparison.ipynb` — consolidate all results into final
publication-style charts and tables for the README/report, and (optionally) build the
Streamlit demo app (`app/streamlit_app.py`) using the saved model.
